# Aula 5 — Nosso primeiro classificador de textos

**De features TF-IDF a uma decisão preditiva com Machine Learning**

Até aqui, aprendemos a transformar texto em dados estruturados, tokens, contagens e pesos TF-IDF.

Agora vamos usar essas features para resolver um problema clássico de Text Intelligence:

> Dada uma nova mensagem, a qual categoria ela pertence?

Nesta aula construiremos um primeiro **baseline supervisionado** usando `TfidfVectorizer` + `MultinomialNB`.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar o que é classificação supervisionada;
- distinguir features de rótulos;
- separar dados em treino e teste;
- construir um pipeline simples com TF-IDF + Naive Bayes;
- gerar predições para novos textos;
- calcular e interpretar acurácia básica;
- entender por que um baseline simples é importante antes de modelos mais sofisticados.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. O problema de classificação

Vamos trabalhar com mensagens rotuladas em três categorias:

- `duvida`
- `reclamacao`
- `elogio`

Cada exemplo contém duas partes essenciais:

```text
texto  → features após representação
rótulo → resposta correta conhecida
```

Como os rótulos são conhecidos durante o treinamento, estamos diante de **aprendizagem supervisionada**.


## 3. Criando um pequeno dataset supervisionado

Execute a próxima célula para criar um conjunto artificial, mas suficiente para observar o fluxo completo de classificação.


In [ ]:
texts = [
    "como altero minha senha",
    "onde vejo minha fatura",
    "posso pagar amanhã",
    "como atualizo meu cadastro",
    "meu pedido não chegou",
    "o atendimento foi péssimo",
    "estou insatisfeito com o serviço",
    "a entrega atrasou novamente",
    "o atendimento foi excelente",
    "fui muito bem atendido",
    "serviço rápido e eficiente",
    "estou satisfeito com o atendimento",
]

labels = [
    "duvida", "duvida", "duvida", "duvida",
    "reclamacao", "reclamacao", "reclamacao", "reclamacao",
    "elogio", "elogio", "elogio", "elogio",
]

print("Documentos:", len(texts))
print("Rótulos:", len(labels))


### O que observar

Temos um conjunto balanceado com quatro exemplos por classe.

Esse dataset é pequeno demais para um sistema real, mas adequado para aprender o pipeline.

**Checkpoint 1:** confirme que cada texto possui exatamente um rótulo correspondente.


## 4. Separando treino e teste

Treinar e avaliar no mesmo conjunto pode produzir uma impressão enganosa de desempenho.

Por isso, vamos reservar parte dos dados para teste usando `train_test_split`.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels,
)

print("Treino:", len(X_train))
print("Teste:", len(X_test))


### Por que `stratify=labels`?

Porque queremos preservar aproximadamente a proporção das classes entre treino e teste.

Isso é especialmente importante quando o dataset é pequeno.


## 5. Construindo o pipeline

Vamos combinar duas etapas:

1. `TfidfVectorizer` transforma texto em features;
2. `MultinomialNB` aprende padrões entre essas features e os rótulos.

`Pipeline` permite encadear as etapas como uma única solução reproduzível.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB()),
])

model.fit(X_train, y_train)
print("Modelo treinado.")


## 6. Fazendo predições

Agora o pipeline consegue receber texto bruto e retornar uma classe prevista.


In [ ]:
predictions = model.predict(X_test)

for text, expected, predicted in zip(X_test, y_test, predictions):
    print("TEXTO    :", text)
    print("ESPERADO :", expected)
    print("PREVISTO :", predicted)
    print("-")


### O que interpretar

Agora temos, para cada documento de teste:

- o texto original;
- o rótulo correto;
- a classe prevista pelo modelo.

É aqui que começa a avaliação real: observar onde o modelo acerta e onde erra.


## 7. Uma primeira métrica: acurácia

Acurácia mede a proporção de previsões corretas entre todas as previsões realizadas.

Ela é uma boa primeira referência, mas não deve ser usada sozinha em problemas reais.


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print("Acurácia:", round(accuracy, 3))


### Cuidado com a interpretação

Nosso conjunto é muito pequeno. Uma única previsão correta ou incorreta altera bastante a acurácia.

Portanto, o objetivo desta aula **não é obter uma boa métrica**, mas compreender o pipeline completo.

Em aulas futuras veremos métricas mais informativas, como precisão, recall, F1-score e matriz de confusão.


## 8. Testando uma nova mensagem

Vamos agora usar o modelo em textos que não fazem parte do dataset original.


In [ ]:
new_messages = [
    "não consigo acessar minha conta",
    "o suporte resolveu tudo rapidamente",
    "estou muito irritado com o atraso",
]

new_predictions = model.predict(new_messages)

for text, prediction in zip(new_messages, new_predictions):
    print(f"{prediction:12} | {text}")


## 9. Por que começar com um baseline simples?

Antes de transformers, embeddings ou LLMs, um baseline clássico ajuda a responder:

- o problema é aprendível com os dados disponíveis?
- quais classes são mais difíceis?
- quanto uma abordagem mais complexa realmente melhora?
- qual é o custo adicional dessa melhoria?

Um modelo simples não é sinônimo de modelo ruim. Ele é uma **referência de comparação**.


## 10. Exercício guiado

Construa um classificador simples para o dataset abaixo:

```python
exercise_texts = [
    "quero saber o prazo",
    "como faço para cancelar",
    "serviço horrível",
    "estou muito decepcionado",
    "atendimento perfeito",
    "gostei muito do suporte",
]

exercise_labels = [
    "duvida", "duvida",
    "reclamacao", "reclamacao",
    "elogio", "elogio",
]
```

Seu código deve:

1. criar um pipeline com `TfidfVectorizer` e `MultinomialNB`;
2. treinar o pipeline;
3. prever a classe de `"o atendimento foi muito ruim"`;
4. exibir a classe prevista.


In [ ]:
# Escreva sua solução aqui.

exercise_texts = [
    "quero saber o prazo",
    "como faço para cancelar",
    "serviço horrível",
    "estou muito decepcionado",
    "atendimento perfeito",
    "gostei muito do suporte",
]

exercise_labels = [
    "duvida", "duvida",
    "reclamacao", "reclamacao",
    "elogio", "elogio",
]

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q5.hint()` e `q5.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q5 = TILExercise(
    hint_text=(
        "Use **scikit-learn**. As peças principais são `Pipeline`, `TfidfVectorizer` e `MultinomialNB`. "
        "Depois de criar o pipeline, use `.fit(exercise_texts, exercise_labels)` e `.predict([...])`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "from sklearn.pipeline import Pipeline\n"
        "from sklearn.feature_extraction.text import TfidfVectorizer\n"
        "from sklearn.naive_bayes import MultinomialNB\n\n"
        "exercise_model = Pipeline([\n"
        "    ('tfidf', TfidfVectorizer()),\n"
        "    ('classifier', MultinomialNB()),\n"
        "])\n\n"
        "exercise_model.fit(exercise_texts, exercise_labels)\n"
        "prediction = exercise_model.predict(['o atendimento foi muito ruim'])\n"
        "print('Classe prevista:', prediction[0])\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q5.hint() ou q5.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q5.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q5.solution()


## 11. Reprodutibilidade

- linguagem: Python;
- bibliotecas: `scikit-learn`;
- representação: TF-IDF;
- modelo: `MultinomialNB`;
- divisão treino/teste: `random_state=42`;
- acelerador: CPU;
- internet: desabilitada;
- dataset externo: nenhum.


## 12. Resumo

Nesta aula, você aprendeu que:

- classificação supervisionada usa exemplos com rótulos conhecidos;
- treino e teste cumprem papéis diferentes;
- TF-IDF pode alimentar um classificador tradicional;
- `MultinomialNB` oferece um baseline simples e eficiente para texto;
- `Pipeline` encadeia representação e modelo;
- acurácia é apenas uma primeira métrica;
- um baseline ajuda a justificar — ou questionar — o uso de modelos mais complexos.

### Ideia principal

```text
Primeiro construa uma referência simples e mensurável.
Depois pergunte se a complexidade adicional realmente vale a pena.
```

**Fim da Aula 5.**
